In [25]:
with open ('../data/shakespeare.txt', 'r', encoding="utf-8") as FILE:
    text = FILE.read()

In [26]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



### Get Character Sets

In [27]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size)
print(f'[{''.join(chars)}]')

65
[
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz]


### Tokenization

In [28]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda e: [stoi[c] for c in e]
decode = lambda d: ''.join ([itos[i] for i in d])

print(encode("hii there!"))
print(decode(encode("hii there!")))

[46, 47, 47, 1, 58, 46, 43, 56, 43, 2]
hii there!


In [29]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])
print(decode(data[:1000].tolist()))

torch.Size([1115393]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [30]:
# now split data into train and test/validation datasets 90-10 split
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [31]:
# Define Block / Training Chunk size
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [32]:
# input vector to transformer
x = train_data[:block_size]
# output vector to transformer
y = train_data[1:block_size+1]

for i in range(block_size):
    context = x[:i+1] # first i elements
    target = y[i]     # desired output given the context - used for self attention in this case
    print(f"when contex={context}, expected_output={target}")

when contex=tensor([18]), expected_output=47
when contex=tensor([18, 47]), expected_output=56
when contex=tensor([18, 47, 56]), expected_output=57
when contex=tensor([18, 47, 56, 57]), expected_output=58
when contex=tensor([18, 47, 56, 57, 58]), expected_output=1
when contex=tensor([18, 47, 56, 57, 58,  1]), expected_output=15
when contex=tensor([18, 47, 56, 57, 58,  1, 15]), expected_output=47
when contex=tensor([18, 47, 56, 57, 58,  1, 15, 47]), expected_output=58


In [33]:
# Batch dimensions - Used for faster implementation by capable GPUs
import time
import torch
seed = 133785 # time.time_ns() # conventionally time is used to create a larger rannge of data
torch.manual_seed = seed
batch_size = 4 # number of independent batches running in parallel
block_size = 8

def get_batch(split):
    """
    generate small batch of data inputs of x and y
    """
    data = train_data if (split=='train') else val_data
    x_idx = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in x_idx])
    y = torch.stack([data[i+1:i+block_size+1] for i in x_idx])
    return x, y

xb, yb = get_batch('train')
print(f"inputs: {xb.shape}\n {decode(xb[0].tolist())}")
print(f"outputs: {yb.shape}\n {decode(yb[0].tolist())}")

print("---------")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target  = yb[b, t]

        print(f"when context={context.tolist()}, target={target}")


inputs: torch.Size([4, 8])
 his outw
outputs: torch.Size([4, 8])
 is outwa
---------
when context=[46], target=47
when context=[46, 47], target=57
when context=[46, 47, 57], target=1
when context=[46, 47, 57, 1], target=53
when context=[46, 47, 57, 1, 53], target=59
when context=[46, 47, 57, 1, 53, 59], target=58
when context=[46, 47, 57, 1, 53, 59, 58], target=61
when context=[46, 47, 57, 1, 53, 59, 58, 61], target=39
when context=[57], target=11
when context=[57, 11], target=0
when context=[57, 11, 0], target=13
when context=[57, 11, 0, 13], target=52
when context=[57, 11, 0, 13, 52], target=42
when context=[57, 11, 0, 13, 52, 42], target=6
when context=[57, 11, 0, 13, 52, 42, 6], target=1
when context=[57, 11, 0, 13, 52, 42, 6, 1], target=44
when context=[57], target=43
when context=[57, 43], target=1
when context=[57, 43, 1], target=59
when context=[57, 43, 1, 59], target=52
when context=[57, 43, 1, 59, 52], target=58
when context=[57, 43, 1, 59, 52, 58], target=53
when context=[57

In [34]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token reads the logits/(embedded vectors) for the next token from a LUT
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(x) # (B,T,C) or (Batch, Time, Channels) or (BatchSize, BlockSize, NumTokens)
        
        if targets is None:
            loss = None
        else: 
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, x, max_new_tokens):
        for _ in range(max_new_tokens):
            # compute predictions
            logits, _ = self(x)
            # focus only on last time step
            logits = logits[:, -1, :]
            # softmax
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            x_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # concatenate generated output
            x = torch.cat((x, x_next), dim=1)
        return x



m = BigramLanguageModel(vocab_size=vocab_size)
logits, loss = m(xb, yb)
print(loss.shape)
print(loss)
print(logits.shape)

print(decode(m.generate(x=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([])
tensor(5.0657, grad_fn=<NllLossBackward0>)
torch.Size([32, 65])



DYttaxw-.P?ypmUM;$uXgIUEVYr 3MGCd3zNxVhYe$dVY.UJAXVD
.jHwvrpwqyxA?I'KO!dGI.rkxRvd:HgbHzCrCaQdtAcEMMx


In [35]:
optimizer  = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [36]:
batch_size = 64
num_iter   = 10000

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
for epoch in range(num_iter):
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    print(loss.item())


cuda
4.7356462478637695
4.731624126434326
4.738181114196777
4.772192478179932
4.7089924812316895
4.852391719818115
4.670811653137207
4.754972457885742
4.7360053062438965
4.725597858428955
4.734475612640381
4.783197402954102
4.707330703735352
4.720423698425293
4.728402137756348
4.744045734405518
4.7024030685424805
4.7791900634765625
4.701647758483887
4.695777416229248
4.756103038787842
4.690793991088867
4.739369869232178
4.724406719207764
4.706929683685303
4.742215633392334
4.6801228523254395
4.7416462898254395
4.697238922119141
4.652811050415039
4.662607669830322
4.678236961364746
4.731812477111816
4.648283958435059
4.785030364990234
4.672497272491455
4.737208843231201
4.6804728507995605
4.642823219299316
4.722811222076416
4.7290425300598145
4.698254108428955
4.697686672210693
4.78759765625
4.757047653198242
4.775721549987793
4.683781147003174
4.653940200805664
4.6400370597839355
4.677266597747803
4.658132553100586
4.747218608856201
4.61729097366333
4.6684889793396
4.670616626739502
4.

In [37]:
print(decode(m.generate(torch.zeros(1, 1, dtype=torch.long), max_new_tokens=500)[0].tolist()))


Ho illlyero hank akis.
bug:
S: rar Bo thertyor
ARDoffoley, t be acisuze:


IThcobe, at-fe:
S: chandaleme, se we t tr'ronouthest hitha is.
thoy, press. peout as lllendond:
Hanewe t.

Thas as t ind tofugice.
Bou eno we, lxincl'd Be lpanethero ris hand fr:
I d n DWhe rjoun;
KEro ar hendy gneree thist, tshale,
ST:
Thive lehe s urathovaye maremfoweat thea ake MENut CHAug we burontoueidothat!
aked he. w, at, et ar n ie drifr sthilan d
NUME hJatord thtil s
Tiot I atithamby, w wior. creotha pe huecrno's


### Self Attention

In [38]:
torch.manual_seed = seed

B, T, C = 4, 8, 2 # Batch, Time, Channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [39]:
"""
Inefficent Implementation
"""

# xbow => x - bag of words (convention when averaging up tensors)
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev, 0)

In [40]:
xbow[0]

tensor([[ 1.4516,  1.9868],
        [ 0.9770,  0.1116],
        [ 0.6739,  0.0610],
        [-0.2150,  0.3139],
        [-0.0488,  0.2958],
        [-0.2789,  0.2806],
        [-0.4044,  0.0541],
        [-0.3866, -0.2562]])

In [41]:
"""
Efficient Implementation : Matrix multiplication
"""

# Example
torch.manual_seed = seed

a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3,2)).float()
c = a@b
print(f"a=\n---\n{a}")
print(f"b=\n---\n{b}")
print(f"c=\n---\n{c}")


a=
---
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
---
tensor([[3., 2.],
        [6., 0.],
        [9., 1.]])
c=
---
tensor([[3.0000, 2.0000],
        [4.5000, 1.0000],
        [6.0000, 1.0000]])


In [42]:
B, T, C = 4, 8, 2 # Batch, Time, Channels
x = torch.randn(B,T,C)
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei@x
print(wei)
print(xbow2, xbow)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
tensor([[[-1.0760e+00, -9.9793e-01],
         [-1.1724e+00, -5.9503e-01],
         [-9.3049e-01, -5.9212e-01],
         [-7.4935e-01, -9.8022e-01],
         [-5.9018e-01, -5.9568e-01],
         [-2.6905e-01, -2.2223e-01],
         [-1.4586e-03, -2.1103e-01],
         [ 4.5386e-02, -6.0290e-02]],

        [[ 3.4810e+00,  1.3618e-01],
         [ 1.4584e+00,  5.6469e-01],
         [ 9.9924e-01,  1.3526e-01]

In [ ]:
# With softmax
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed = 13357
B, T, C = 4, 8, 32 # Batch, Time, Channels
head_size = 16
x = torch.randn(B, T, C)
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = key(x)
wei = q @ k.transpose(-2, -1)
tril = torch.tril(torch.ones(T, T))
x = torch.randn(B, T, C)
v = value(x)
# wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
# xbow3 = wei @ x
xbow3 = wei @ v
xbow3.shape
# torch.allclose(xbow2, xbow3)
# xbow2[0], xbow2[0]

torch.Size([4, 8, 16])

In [71]:
xbow3

tensor([[[ 4.4052e-01, -2.1437e-04,  3.5867e-01, -3.0107e-01,  5.2036e-01,
          -1.1013e+00,  5.6377e-01,  5.5895e-01,  6.5955e-01, -1.0202e+00,
           7.8062e-01,  5.8780e-01, -5.1684e-02, -7.5211e-01,  7.2787e-01,
          -3.6120e-02],
         [ 1.9926e-01, -2.3728e-01,  3.1023e-01, -1.1560e-02, -4.9289e-01,
          -7.2444e-01, -1.7431e-01,  2.3027e-01, -4.1511e-01, -1.6569e-01,
           2.3670e-01,  5.1514e-01,  7.6407e-02,  4.5948e-01, -2.8395e-01,
           1.5393e-01],
         [-3.1453e-01, -2.6135e-01, -5.6185e-01, -2.4906e-02,  4.0357e-01,
          -1.5025e-01,  4.7717e-01, -1.2137e-01,  1.0085e+00,  1.8870e-01,
          -3.8794e-01,  2.5197e-01,  6.2336e-01,  3.5657e-01,  4.5169e-01,
          -3.0336e-02],
         [-1.7743e-01,  3.6430e-01, -4.0818e-01, -6.2385e-01,  4.5553e-01,
           1.0687e+00,  4.8023e-01, -5.9513e-02, -4.3859e-01, -4.4301e-01,
          -6.2931e-01, -9.8590e-02, -7.9572e-01, -5.5610e-01,  1.8347e-01,
           9.3886e-01],
    